In [ ]:
from pathlib import Path
import time

from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing import InstancePipeline, InstanceParams, PipelineConfig
from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.stations.providers import DonkeyRepublicProvider
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logging.getLogger("sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder").setLevel(logging.INFO)


def main() -> None:
    total_start = time.perf_counter()

    config = PipelineConfig(
        center_lat=46.20604780106243,
        center_lon=6.147834100933916,
        radius_m=1000,
        output_dir=Path("data/sum_nsm_dynamic_pricing/geneva_1km_mon08h"),
        gtfs_path=Path("/Users/rebecamurillo/Documents/INRIA/development/inocs-sum-gtfs-geojson/otp/data/geneva-2026-gtfs.zip"),
        gbfs_path=None,
        seed=42,
        otp_timestamp="2026-03-08T08:00:00.000000Z",
        otp_workers=10
    )
    instance_params = InstanceParams()

    pipeline = InstancePipeline(config=config, gbfs_provider=DonkeyRepublicProvider())

    start = time.perf_counter()
    stations_path = pipeline.prepare_stations()
    print(f"[TIMING] prepare_stations: {time.perf_counter() - start:.2f}s")

    start = time.perf_counter()
    hubs_path = pipeline.build_hubs(stations_path)
    print(f"[TIMING] build_hubs: {time.perf_counter() - start:.2f}s")

    start = time.perf_counter()
    routes_path = pipeline.compute_routes(hubs_path)
    print(f"[TIMING] compute_routes: {time.perf_counter() - start:.2f}s")
    print(f"Starting route quality validation for: {routes_path}")
    validated = pipeline.quality_routes(routes_path)
    print(f"Completed route quality validation for: {routes_path} - Validated: {validated}")

    start = time.perf_counter()
    instance_path = pipeline.generate_instance(instance_params)
    print(f"[TIMING] generate_instance: {time.perf_counter() - start:.2f}s")

    start = time.perf_counter()
    web_demo_path = pipeline.export_web_demo()
    print(f"[TIMING] export_web_demo: {time.perf_counter() - start:.2f}s")

    print(f"Instance generated at: {instance_path}")
    print(f"Web demo data exported to: {web_demo_path}")
    print(f"[TIMING] total: {time.perf_counter() - total_start:.2f}s")


if __name__ == "__main__":
    main()


## Optimization and Metrics Step
Run the exact optimization model on the generated instance and then export summary metrics.

In [ ]:
from pathlib import Path

from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing import (
    InstancePipeline,
    OptimizerParams,
    PipelineConfig,
    run_pipeline,
    InstanceParams,
 )
from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.stations.providers import DonkeyRepublicProvider

output_dir = Path("data/sum_nsm_dynamic_pricing/geneva_1km_mon08h")

config = PipelineConfig(
    center_lat=46.2059149,
    center_lon=6.1438511,
    radius_m=1000,
    output_dir=output_dir,
    gtfs_path=Path("/Users/rebecamurillo/Documents/INRIA/development/inocs-sum-gtfs-geojson/otp/data/geneva-2026-gtfs.zip"),
    gbfs_path=None,
    seed=42,
    otp_timestamp="2026-03-08T08:30:00.000000Z",
 )

pipeline = InstancePipeline(config, gbfs_provider=DonkeyRepublicProvider())

# Optional: run the full instance generation pipeline if no instance exists yet.
instance_candidates = sorted(output_dir.glob("Instance_*.json"))
if instance_candidates:
    instance_path = instance_candidates[-1]
else:
    instance_path = "NONE"

print(f"Using instance: {instance_path}")

params = OptimizerParams(
    max_displayed_alternatives=10,
    P_bar=0.6,
    pt_price=2.0,
    cost_bike=0.6,
    parking_fee=2.8,
    theta=0.5,
    ridership_weight=8.0,
    verbose=False,
    verbose2=False,
)
solution_path = pipeline.optimize(
    params=params,
    instance_path=instance_path,
)

print(f"Optimization results: {solution_path}")

metrics_path = pipeline.summarize_optimization(solution_path, params)
print(f"Optimization metrics: {metrics_path}")
